# Surya OCR Server for SME-GPT

This notebook runs a Surya OCR HTTP server and exposes it via an ngrok tunnel so the SME-GPT backend can call it.

### Before running:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Get a free ngrok account** at https://dashboard.ngrok.com and copy your auth token
3. Paste the token in **Cell 3** below
4. Run all cells top to bottom (Runtime → Run all)
5. Copy the printed URL into your `backend/.env` as `COLAB_OCR_URL=...`

> Keep this tab open while using the backend — closing it kills the server.

In [ ]:
# Cell 1 — Install dependencies
# surya-ocr 0.13.1 = the LAST pure-PyTorch release before "Surya v2" (0.14.0+),
# which auto-spawns vLLM/llama-server inside Docker (Colab has no Docker daemon
# -> "docker binary not found").
#
# 0.13.1 requires torch>=2.5.1 (already on Colab) and does NOT depend on
# torchvision, so a PLAIN install leaves Colab's CUDA-matched torch/torchvision
# untouched. DO NOT add --force-reinstall here: it reinstalls torch with a
# generic build and breaks torchvision ("operator torchvision::nms does not
# exist"). If that already happened, do Runtime > Disconnect and delete runtime
# to get a clean VM, THEN run this cell.
!pip uninstall -y surya-ocr
!pip install "surya-ocr==0.13.1" flask pyngrok

import importlib.metadata as _m
import torch, torchvision
print("\n" + "=" * 60)
print(f"surya-ocr   : {_m.version('surya-ocr')}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
try:
    import torchvision.ops
    torchvision.ops.nms(
        torch.tensor([[0.0, 0.0, 1.0, 1.0]]), torch.tensor([0.5]), 0.5
    )
    print("torchvision ops: OK — safe to continue to Cell 2.")
except Exception as e:
    print(f"torchvision ops: BROKEN -> {e}")
    print("  -> Runtime > Disconnect and delete runtime, then run all cells again.")
print("=" * 60)

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. OCR will run on CPU and will be slow.")
    print("Go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
# Cell 3 — Set your ngrok auth token
# Get it free from: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "https://dashboard.ngrok.com/"  # <-- replace this

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok token set.")

In [ ]:
# Cell 4 — Load Surya models (surya-ocr 0.13.1, pure PyTorch, no Docker)
# 0.13.x API:  RecognitionPredictor()  +  DetectionPredictor()
#   predictions = rec([image], [langs], det_predictor)
# (FoundationPredictor / surya.foundation only exists in Surya v2 / 0.14+, which
#  needs Docker — we guard for it so this cell also survives a version bump.)

import inspect
import os
import importlib.metadata as _md
import torch

os.environ.setdefault("TORCH_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
os.environ.setdefault("RECOGNITION_BATCH_SIZE", "16")

from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

try:
    # Surya v2 (0.14+) requires a FoundationPredictor; 0.13.x has no such module.
    from surya.foundation import FoundationPredictor
    foundation_predictor = FoundationPredictor()
    rec_predictor = RecognitionPredictor(foundation_predictor)
except ModuleNotFoundError:
    foundation_predictor = None
    rec_predictor = RecognitionPredictor()

det_predictor = DetectionPredictor()

# Cache the recognition call signature so Cell 5 picks the right argument style
try:
    _rec_call_params = set(inspect.signature(rec_predictor.__call__).parameters.keys())
except Exception:
    _rec_call_params = set()

SURYA_MODE = f"surya-{_md.version('surya-ocr')} ({os.environ.get('TORCH_DEVICE')})"

print(f"RecognitionPredictor call params : {_rec_call_params}")
print(f"Mode                             : {SURYA_MODE}")
print("Surya ready.")

In [ ]:
# Cell 5 — OCR helper

from PIL import Image


def _normalize(v):
    if v is None:
        return None
    return v.tolist() if hasattr(v, "tolist") else list(v)


def _call_rec_predictor(pil_image):
    """Call rec_predictor, adapting to the installed Surya API version."""
    # Surya 0.13.x: rec([images], [langs], det_predictor). langs=None is fine.
    if "det_predictor" in _rec_call_params:
        if "langs" in _rec_call_params:
            return rec_predictor([pil_image], None, det_predictor)
        return rec_predictor([pil_image], det_predictor=det_predictor)

    # Surya v2: run detection first, then pass results into recognition.
    det_results = det_predictor([pil_image])
    if "det_predictions" in _rec_call_params:
        return rec_predictor([pil_image], det_predictions=det_results)

    # Last resort: positional, then no-detection fallback.
    try:
        return rec_predictor([pil_image], det_results)
    except TypeError:
        return rec_predictor([pil_image])


def run_surya_on_image(pil_image: Image.Image) -> dict:
    """Run Surya OCR on a single PIL image. Returns {text, text_lines}."""
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")

    predictions = _call_rec_predictor(pil_image)

    if not predictions:
        return {"text": "", "text_lines": []}

    pred = predictions[0]
    raw_lines = getattr(pred, "text_lines", None) or []

    text_lines = [
        {
            "text"      : getattr(ln, "text", "") or "",
            "confidence": float(getattr(ln, "confidence", 0.0) or 0.0),
            "polygon"   : _normalize(getattr(ln, "polygon", None)),
            "bbox"      : _normalize(getattr(ln, "bbox", None)),
        }
        for ln in raw_lines
    ]

    full_text = "\n".join(
        ln["text"].strip() for ln in text_lines if ln["text"].strip()
    ).strip()

    return {"text": full_text, "text_lines": text_lines}


print("OCR helper ready.")

In [ ]:
# Cell 6 — Flask server
# POST /ocr  — receives multipart images, returns JSON in the format the SME-GPT backend expects.
# GET  /health — liveness check.

import io
from threading import Thread
from flask import Flask, request, jsonify

app = Flask(__name__)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "engine": "surya", "mode": SURYA_MODE})


@app.route("/ocr", methods=["POST"])
def ocr():
    try:
        orig_files = request.files.getlist("orig")
        p_files    = request.files.getlist("p_img")
        m_files    = request.files.getlist("m_img")

        if not orig_files:
            return jsonify({"success": False, "error": "No 'orig' images received."}), 400

        versions = {
            "orig": {"pages": [], "text": ""},
            "P"   : {"pages": [], "text": ""},
            "M"   : {"pages": [], "text": ""},
        }
        failures = {}

        for version_name, files in [("orig", orig_files), ("P", p_files), ("M", m_files)]:
            page_texts = []
            for page_idx, upload in enumerate(files, start=1):
                try:
                    pil_img = Image.open(io.BytesIO(upload.read()))
                    result  = run_surya_on_image(pil_img)

                    versions[version_name]["pages"].append({
                        "page"      : page_idx,
                        "text"      : result["text"],
                        "text_lines": result["text_lines"],
                    })
                    page_texts.append(result["text"].strip())

                except Exception as e:
                    failures[f"{version_name}_page_{page_idx}"] = str(e)

            # Merge pages into a single text blob
            parts = []
            for i, txt in enumerate(page_texts, start=1):
                if txt:
                    parts.append(f"=== PAGE {i} ===\n{txt}" if len(page_texts) > 1 else txt)
            versions[version_name]["text"] = "\n\n".join(parts).strip()

        return jsonify({
            "success" : True,
            "engine"  : "colab_surya",
            "versions": versions,
            "failures": failures,
        })

    except Exception as e:
        import traceback
        return jsonify({"success": False, "error": str(e), "trace": traceback.format_exc()}), 500


# Run Flask in a background thread so the cell doesn't block
flask_thread = Thread(target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False), daemon=True)
flask_thread.start()
print("Flask server listening on port 5000.")

In [ ]:
# Cell 7 — Open ngrok tunnel and print the URL to copy

try:
    ngrok.kill()  # close any leftover tunnels from previous runs
except Exception:
    pass

tunnel = ngrok.connect(5000, bind_tls=True)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print("Colab OCR server is live!")
print("="  * 60)
print(f"\nURL  :  {public_url}")
print(f"\nPaste this into backend/.env:")
print(f"  COLAB_OCR_URL={public_url}")
print("\n" + "=" * 60)
print("Keep this tab open. Closing it kills the server.")

In [ ]:
# Cell 8 — Optional: quick self-test
# Run this after pasting the URL into .env to confirm the server is reachable.

import requests
resp = requests.get(f"{public_url}/health", timeout=10)
print(resp.json())